In [22]:
import re
import pandas as pd

In [23]:
def process_eko_transactions():

    data = pd.read_excel("data/eko_transaction.xlsx", dtype={"Number": str})
    cards = pd.read_excel("data/cards.xlsx", dtype={"Number": str})

    data["Material"] = data["Material"].apply(
        lambda x: re.sub(r'^\d+\s*', '', str(x))
    )

    data["Name"] = data["Name"].astype(str).str.replace(" ", "", regex=False)

    data = data[
        [
            "Plant",
            "Name",
            "Number",
            "Material",
            "Date",
            "Bill.qty",
            "Bill.qty.1",
            "FinPr",
            "Tot Amount",
            "Auth.time",
        ]
    ]

    data["Number"] = data["Number"].str.replace(".0", "", regex=False).str.strip()
    cards["Number"] = cards["Number"].str.replace(".0", "", regex=False).str.strip()

    data = data.merge(
        cards[["Number", "Company"]],
        on="Number",
        how="left"
    )

    data = data[data["Number"].str.len() >= 17]

    with pd.ExcelWriter("result/eko_transactions.xlsx") as writer:

        for company, df in data.groupby("Company", dropna=False):
            sheet_name = str(company)[:31]
            df.to_excel(writer, sheet_name=sheet_name, index=False)

    return data

In [24]:
data = process_eko_transactions()